# 11 — Zbritja e gradientit dhe optimizimi numerik

## Konteksti dhe problemi shkencor

Nga një funksion kuadratik te regresioni linear, studiojmë gradient descent si metodë të përgjithshme optimizimi dhe jo si teknikë ekskluzive e rrjetave nervore.

### Objektivat e të nxënit

- të derivoni dhe implementoni gradient descent;
- të kuptoni learning rate;
- të lidhni scaling me kondicionimin;
- të dalloni full-batch, SGD dhe minibatch.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Një minimum kuadratik

In [ ]:
def f(t): return t*t
def grad(t): return 2*t
for eta in [.01,.1,.8,1.1]:
 t=4.; hist=[t]
 for _ in range(25): t-=eta*grad(t); hist.append(t)
 plt.plot(hist,marker='o',ms=2,label=f'η={eta}')
plt.legend(); plt.show()

## 2. Kushti analitik

Për $L=\theta^2$, $\theta_{n+1}=(1-2\eta)\theta_n$. Konvergjenca kërkon $|1-2\eta|<1$.

## 3. Regresion linear me gradient descent

In [ ]:
df=pd.read_csv(DATA/'gradient_regression.csv'); X=df[['x']].to_numpy(); y=df.y.to_numpy(); from src.optimization.manual import linear_gd; w,h=linear_gd(X,y,lr=.03,n_iter=1200); print(w); plt.plot(h); plt.yscale('log'); plt.show()

## 4. Krahasim me OLS

In [ ]:
m=LinearRegression().fit(X,y); print([m.intercept_,m.coef_[0]],w)

## 5. Sipërfaqe e humbjes

In [ ]:
bs=np.linspace(-2,1,120); ms=np.linspace(0,3,120); Z=np.empty((len(bs),len(ms)))
for i,b in enumerate(bs):
 for j,a in enumerate(ms): Z[i,j]=np.mean((b+a*X[:,0]-y)**2)
plt.contour(ms,bs,Z,levels=25); plt.xlabel('slope'); plt.ylabel('intercept'); plt.show()

## 6. Scaling dhe kondicionimi

In [ ]:
rng=np.random.default_rng(0); X2=np.c_[rng.normal(size=500),1000*rng.normal(size=500)]; y2=2*X2[:,0]-.003*X2[:,1]+rng.normal(0,.2,500);
# Përdorim SGDRegressor vetëm për të demonstruar ndjeshmërinë praktike.
from sklearn.linear_model import SGDRegressor
for use_scale in [False,True]:
 mdl=make_pipeline(StandardScaler(),SGDRegressor(max_iter=3000,tol=1e-7,random_state=0)) if use_scale else SGDRegressor(max_iter=3000,tol=1e-7,random_state=0); mdl.fit(X2,y2); print(use_scale,mdl.score(X2,y2))

## 7. Zbritja stokastike

In [ ]:
rng=np.random.default_rng(1); A=np.c_[np.ones(len(X)),X]; ww=np.zeros(2); losses=[]
for epoch in range(30):
 for i in rng.permutation(len(X)):
  r=A[i]@ww-y[i]; ww-=.01*2*r*A[i]
 losses.append(np.mean((A@ww-y)**2))
print(ww); plt.plot(losses,marker='o'); plt.show()

## 8. Minibatch

In [ ]:
def minibatch_gd(A,y,lr=.02,batch=16,epochs=100,seed=0):
 rng=np.random.default_rng(seed); w=np.zeros(A.shape[1]); hist=[]
 for _ in range(epochs):
  for start in range(0,len(A),batch):
   idx=rng.permutation(len(A))[start:start+batch]; r=A[idx]@w-y[idx]; w-=lr*2*A[idx].T@r/len(idx)
  hist.append(np.mean((A@w-y)**2))
 return w,hist
for b in [1,8,32,len(A)]:
 _,hh=minibatch_gd(A,y,batch=b); plt.plot(hh,label=f'b={b}')
plt.yscale('log'); plt.legend(); plt.show()

## 9. Ushtrime

- Derivoni gradientin e MSE.
- Gjeni eta kritik për një funksion $a\theta^2/2$.
- Krahasoni batch sizes dhe kohën.
- Provoni schedule $\eta_t=\eta_0/(1+kt)$.

## 10. Shembull i plotë

Përshtatni të njëjtin regresion me OLS, full-batch GD dhe SGD; krahasoni koeficientët, kohën dhe trajektoren e humbjes.

## 11. Playground

Ndryshoni korelacionin e veçorive dhe vëzhgoni se si ndryshon kondicionimi dhe forma e kontureve të humbjes.